In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")[:10]

In [2]:
OPENAI_API_KEY

'sk-proj-7I'

In [3]:
# 필수 import v1.0
from langchain_openai.chat_models.base import ChatOpenAI
from langchain_openai.llms.base import OpenAI
from langchain_core.output_parsers.base import BaseOutputParser
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.prompts.chat import ChatMessagePromptTemplate, ChatPromptTemplate

In [5]:
chat = ChatOpenAI()

In [8]:
chat.invoke("호날두와 메시가 싸우면 누가 이길까")

AIMessage(content='호날두와 메시는 둘 다 세계적인 축구 스타이기 때문에 누가 이긴다고 단정지을 수는 없습니다. 하지만 일반적으로 호날두는 체력과 파워가 강하고 물리적인 부분에서 우세하며, 메시는 기술과 민첩성, 스트라이킹 능력이 뛰어나다는 평가를 받습니다. 따라서 이 둘의 대결은 매우 균형을 이루고 있으며 결과는 어느 쪽이 승리할지 예측하기 어렵습니다. 이경기로 나가면 축구 팬들에게는 엄청난 즐거움이 될 것입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 222, 'prompt_tokens': 29, 'total_tokens': 251, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Debn6XpHJvv7m60RtRHjbBs7vOqus', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e1b14-0c59-7291-8fb7-a2dcc5d90c4b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 29, 'output_tokens': 222, 'total_tokens': 251, 'input_token_details': {'aud

In [9]:
# 이 질문은 가상의 상황이므로 정답을 제시하기 어렵습니다. 
# 호날두와 메시는 모두 축구계의 전설적인 선수들로, 각자의 장점과 능력을 가지고 있습니다. 
# 그들이 싸움을 벌인다면 양쪽에서 피해를 입을 가능성이 크고, 해결 방법은 상호 존중하며 문제를 해결하는 방법일 것입니다.
# 그들이 우호적으로 해결 방법을 찾는 것이 가장 이상적이며, 
# 그 둘의 충돌은 잘못된 방향으로 나아가는 것이라 생각할 뿐입니다.

# 호날두와 메시는 둘 다 세계적인 축구 스타이기 때문에 누가 이긴다고 단정지을 수는 없습니다. 
# 하지만 일반적으로 호날두는 체력과 파워가 강하고 물리적인 부분에서 우세하며, 
# 메시는 기술과 민첩성, 스트라이킹 능력이 뛰어나다는 평가를 받습니다. 
# 따라서 이 둘의 대결은 매우 균형을 이루고 있으며 결과는 어느 쪽이 승리할지 예측하기 어렵습니다. 
# 이경기로 나가면 축구 팬들에게는 엄청난 즐거움이 될 것입니다.

## temperature(창의력)
- 0.0 ~ 0.3: 결정론 - 요약, 추출, 일관된, 정확성
- 0.5 ~ 0.7: 균형적 - 적당히 자연스러움, 유연성
- 0.8 ~ 1.0+: 무작위 - 창의적이고 예측 불가능

In [15]:
chat = ChatOpenAI(temperature=1.0)
result = chat.invoke("하늘은 무슨 색이니")
result.content

'하늘의 색은 다양하게 변할 수 있지만 일반적으로 푸른색을 띠고 있습니다. 특히 맑은 날에는 하늘이 맑고 푸르스름한 색을 내보이는 것이 보편적입니다. 그러나 일출이나 일몰 시간대에는 주로 붉은, 주황색이 혼합된 아름다운 색조를 보여줄 수도 있습니다. 또한 날씨나 계절에 따라 구름의 어두운 그림자가 들어가는 등 하늘의 색은 변화무쌍하고 아름다운 것으로 유명합니다.'

In [20]:
class NewLineOutputParser(BaseOutputParser):
    # 반드시 parse
    def parse(self, text):
        lines = text.split("\n")
        return [line.lstrip("-123456789.").strip() for line in lines]

In [21]:
newline_parser = NewLineOutputParser()

In [22]:
newline_parser.parse("""- 1. 햄버거\n 2. 떡볶이\n- 3. 치킨""")

['1. 햄버거', '2. 떡볶이', '3. 치킨']

In [24]:
template = ChatPromptTemplate.from_messages([
    ("system", """
            리스트를 생성하는 기계입니다.
        요청한 모든 리스트에 개수는 최대 {max_length}개 까지만 목록으로 표시하세요.
        그 이상 초과되는 리스트는 답변하지 마세요.
    """),
    ("human", "{question}")
])

prompt = template.format_messages(
    max_length = 5,
    question="AI를 잘하기 위해 어떤 것 부터 공부해야하는가?"
)

## 체인(Chain) 생성

- "|": 파이프 연산자로 체인을 만든다

In [28]:
#list
first_chain = template | chat | NewLineOutputParser()

In [29]:
chain_result = first_chain.invoke({
    "max_length": 5,
    "question": "AI를 잘하려면 어떤 것부터 공부해야하는가?"
})

In [30]:
chain_result

['머신 러닝 기초: 데이터 전처리, 특성 공학, 모델 선택 등을 이해하는 것이 중요합니다.',
 '수학적 기반: 선형 대수, 확률과 통계, 미적분학 등을 공부하여 머신 러닝 알고리즘 이해에 도움이 됩니다.',
 '프로그래밍 언어: Python이 대표적으로 사용되며, TensorFlow나 PyTorch와 같은 라이브러리 사용법을 익히는 것이 도움이 됩니다.',
 '머신 러닝 알고리즘: 지도 학습, 비지도 학습, 강화 학습 등의 다양한 알고리즘을 이해하고 실제 데이터에 적용할 수 있는 능력을 길러야 합니다.',
 '실습과 프로젝트: Kaggle 등의 경진대회나 실제 데이터셋을 활용한 프로젝트를 통해 실력을 향상시키는 것이 좋습니다.']

In [31]:
#RunnableSequence
#이전 단계의 출력이 다음 단계의 입력으로 자동 전달되는 파이프라인 객체
print(type(first_chain))

<class 'langchain_core.runnables.base.RunnableSequence'>


## 1. 템플릿 생성

In [49]:
template = ChatPromptTemplate.from_messages([
    ("system", """
        당신은 세계적인 수준의 여행가이드입니다.
        사람들이 좋아하는 여행 장소를 많이 알고 있습니다.
        설명 없이 지역 명소 이름만 목록으로 {max_length}개 까지 답변하세요.
        목록의 개수가 초과하는 것은 답변하지 마세요.
    """),
    ("human", """
        {place} 여행 장소 추천해줘!
    """)
])

In [50]:
second_chain = template | chat

In [51]:
second_chain.invoke({
    "max_length" : 3,
    "place" : "이탈리아"
})

AIMessage(content='1. 로마\n2. 피렌체\n3. 베니스', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 134, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DecTriKyNwqvZL91NZiiRXF2k7VAK', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e1b3c-8549-7613-b2a9-68b5407a816a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 134, 'output_tokens': 21, 'total_tokens': 155, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
# 저녁 메뉴 추천(양식, 중식, 한식 선택할 수 있도록)
# 체인을 생성 후 결과를 출력

In [63]:
dinner_template = ChatPromptTemplate.from_messages([
    ("system", """
        당신은 적당한 수준의 요리사입니다.
        당신은 저녁 메뉴를 추천하여 요리해주는 식당을 운영하고 있습니다.
        손님에게 양식, 중식, 한식 중 하나를 선택하도록 하고,
        메뉴를 {max_length}개 까지 추천하세요.
        목록의 개수를 초과하여 답변하지 마세요.
    """),("human", """
        {dinner} 메뉴 추천해줘.
    """)
])

In [64]:
third_chain = dinner_template | chat

In [65]:
food_result = third_chain.stream({
    "max_length" : 1,
    "dinner" : "양식"
})

In [66]:
for chunk in food_result:
    print(chunk.content, end="", flush=True)

고르곤졸라 크림 파스타를 추천해드립니다. 재료를 확인하여 신선하고 맛있는 요리를 준비해보세요.

## .stream()

In [67]:
# 형태로 딕셔너리
# 최근 급등한 주식 리스트 10개

In [78]:
template = ChatPromptTemplate.from_messages([
    ("system", """
        넌 잘나가는 주식의 트레이더야.
        사용자가 주식과 관련된 질문을 하면 {max_length}개의 주식을 추천해줘.

        반드시 아래의 JSON 형식으로 답변해줘.
        출력 형식:
        {{
            "one": "tesla",
            "two": "samsung",
            ...
            "five": "apple"
        }}
    """),
    ("human", "{question}"),
])

In [79]:
stock_chain = template | chat

In [80]:
stock_result = stock_chain.invoke({
    "max_length": 5,
    "question": "최근 급등한 미국 주식 추천해줘"
})

In [81]:
import json

json.loads(stock_result.content)

{'one': 'GameStop',
 'two': 'AMC Entertainment',
 'three': 'Tesla',
 'four': 'Zoom Video Communications',
 'five': 'NIO Inc.'}

In [82]:
class JsonOutputParser(BaseOutputParser):

    def parse(self, text):
        return json.loads(text)

In [88]:
chat = ChatOpenAI(temperature=0)

stock_chain = template | chat | JsonOutputParser()

In [93]:
stock_result = stock_chain.invoke({
    "max_length": 5,
    "question": "최근 급등한 미국 주식 추천해줘"
})

In [95]:
stock_result

{'one': 'Tesla',
 'two': 'Zoom Video Communications',
 'three': 'Moderna',
 'four': 'Peloton',
 'five': 'Shopify'}